In [2]:
from google.colab import userdata
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1 — Installation des dépendances
!pip install -q \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    beautifulsoup4 \
    accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 72.8 MB/s eta 0:00:00:00:0100:01


In [ ]:
# =========================
# Reranker — CrossEncoder
# =========================

from sentence_transformers import CrossEncoder
import torch

print("🔁 Initialisation du reranker (CrossEncoder)...")

def load_reranker(model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
    """
    Charge un reranker CrossEncoder avec gestion automatique GPU / CPU.
    Retourne None si le chargement échoue (fallback propre).
    """

    try:
        device = "cuda" if USE_GPU and torch.cuda.is_available() else "cpu"

        reranker = CrossEncoder(
            model_name,
            device=device,
            max_length=512    # 🔒 limite mémoire, suffisant pour QA
        )

        print(f"✅ Reranker chargé sur {device.upper()} : {model_name}")
        return reranker

    except Exception as e:
        print("⚠️ Impossible de charger le reranker.")
        print(f"🛠️ Détail : {e}")
        print("➡️ Le système continuera sans reranking (FAISS seul).")
        return None


# Chargement effectif
reranker = load_reranker()

2026-01-14 20:49:28.652737: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768423768.836400      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768423768.890827      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768423769.318605      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768423769.318651      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768423769.318654      55 computation_placer.cc:177] computation placer alr

🔁 Initialisation du reranker (CrossEncoder)...
⚠️ Impossible de charger le reranker.
🛠️ Détail : name 'USE_GPU' is not defined
➡️ Le système continuera sans reranking (FAISS seul).


In [ ]:
# =========================
# 2 — Imports & paramètres globaux
# =========================

# ----- Standard library -----
import os
import sys
import pickle
from typing import List, Dict

# ----- Calcul scientifique -----
import numpy as np

# ----- GPU / Deep Learning -----
import torch

# ----- Traitement documents -----
import fitz  # PyMuPDF (PDF)
from bs4 import BeautifulSoup  # HTML

# ----- Recherche vectorielle -----
import faiss

# ----- Embeddings & reranking -----
from sentence_transformers import SentenceTransformer, CrossEncoder

# ----- LLM / Hugging Face -----
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

# =========================
# Paramètres globaux
# =========================

# Mode debug (logs détaillés)
DEBUG = False

# Seuils & constantes de sécurité
MIN_TEXT_LENGTH = 50         # Ignore les textes trop courts
MIN_QUERY_LENGTH = 3         # Ignore les questions trop vagues

# Mémoire & performances
DEFAULT_BATCH_GPU = 8
DEFAULT_BATCH_CPU = 4

# Répertoire de travail
PROJECT_ROOT = "/content"
FAISS_INDEX_DIR = os.path.join(PROJECT_ROOT, "faiss_index_cnrs")

# Création du dossier FAISS si nécessaire
os.makedirs(FAISS_INDEX_DIR, exist_ok=True)

print("✅ Imports et paramètres globaux chargés.")# =========================
# 1 — Installation des dépendances (robuste)
# =========================

# Mise à jour pip (important sur Colab)
!pip install -U pip

# Dépendances principales (versions stables et testées)
!pip install -q \
    torch \
    transformers==4.39.3 \
    sentence-transformers==2.6.1 \
    accelerate==0.27.2 \
    faiss-cpu==1.7.4 \
    pymupdf==1.23.8 \
    beautifulsoup4==4.12.3


✅ Imports et paramètres globaux chargés.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:0000:010:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2)
ERROR: No matching distribution found for faiss-cpu==1.7.4


In [ ]:
# =========================
# PARAMÈTRES (VERSION STABLE)
# =========================


DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full/cleaned_json_full"


MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
load_in_4bit = True

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# Chunking (OK pour JSON structuré)
CHUNK_SIZE = 400
OVERLAP = 100

# Retrieval
TOP_K = 8
SIMILARITY_THRESHOLD = 0.30

# 🔒 Sécurité mémoire LLM
MAX_CHUNKS_FOR_LLM = 3        #  CRITIQUE
MAX_CONTEXT_CHARS = 700       # réduit (anti-OOM)

# Génération
MAX_NEW_TOKENS = 200
TEMPERATURE = 0.2
REPETITION_PENALTY = 1.1


In [ ]:
# =========================
# 3 — Détection GPU automatique
# =========================

import torch

def can_use_gpu(min_free_gb: float = 4.0, verbose: bool = True) -> bool:
    """
    Détermine si le GPU peut être utilisé en toute sécurité.

    Paramètres :
    - min_free_gb : mémoire GPU libre minimale requise (en Go)
    - verbose : affiche des informations détaillées

    Retour :
    - True si le GPU est utilisable
    - False sinon (fallback CPU)
    """

    # 1️⃣ CUDA disponible ?
    if not torch.cuda.is_available():
        if verbose:
            print("⚠️ CUDA non disponible → mode CPU")
        return False

    try:
        # 2️⃣ Mémoire GPU disponible
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / (1024 ** 3)
        total_gb = total_bytes / (1024 ** 3)

        # 3️⃣ Informations GPU
        device_name = torch.cuda.get_device_name(0)

        if verbose:
            print(f"🖥️ GPU détecté : {device_name}")
            print(f"💾 Mémoire GPU libre : {free_gb:.2f} Go / {total_gb:.2f} Go")

        # 4️⃣ Seuil minimal requis
        if free_gb < min_free_gb:
            if verbose:
                print(
                    f"⚠️ Mémoire GPU insuffisante "
                    f"(min requis : {min_free_gb} Go) → mode CPU"
                )
            return False

        return True

    except Exception as e:
        if verbose:
            print(f"❌ Erreur lors de la détection GPU : {e}")
            print("➡️ Fallback CPU")
        return False


# =========================
# Sélection automatique du mode
# =========================

USE_GPU = can_use_gpu(min_free_gb=4.0)

print(
    f"\n🔍 Mode sélectionné : "
    f"{'🟢 GPU' if USE_GPU else '🟡 CPU'}\n"
)


🖥️ GPU détecté : Tesla T4
💾 Mémoire GPU libre : 14.64 Go / 14.74 Go

🔍 Mode sélectionné : 🟢 GPU



In [ ]:
# =========================
# 4 — Chargement et préparation des documents
#     (DOSSIER JSON STRUCTURÉ – VERSION FINALE)
# =========================

import os
import json
import re


# ---------- Nettoyage ----------
def clean_text(text: str) -> str:
    text = text.replace("\x00", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


# ---------- Normalisation ----------
def normalize_text(text: str) -> str:
    text = text.replace("•", "- ")
    text = text.replace("–", "- ")
    text = text.replace("—", "- ")
    text = text.replace("* ", "- ")
    return text


# ---------- Construction de chunks logiques ----------
def build_chunks_from_poste(base_meta: dict, poste: dict):
    """
    Crée des chunks séparés pour mission, activités, compétences, contexte.
    """
    chunks = []

    def add_chunk(label, content):
        if content and len(content.strip()) > 50:
            chunks.append({
                "text": f"{label} : {normalize_text(clean_text(content))}",
                **base_meta
            })

    add_chunk("Mission", poste.get("mission"))
    add_chunk("Activités", poste.get("activites"))
    add_chunk("Compétences", poste.get("competences"))
    add_chunk("Contexte", poste.get("contexte"))

    return chunks


# =========================
# CONSTRUCTION DU CORPUS
# =========================

documents = []

print("📂 Chargement des documents JSON structurés...")

for filename in os.listdir(DATA_DIR):
    if not filename.lower().endswith(".json"):
        continue

    path = os.path.join(DATA_DIR, filename)

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Métadonnées communes au concours
        base_meta_common = {
            "source": filename,
            "bap": data.get("bap"),
            "grade": data.get("grade"),
            "concours_label": data.get("concours_label"),
            "concours_num": data.get("concours_num"),
            "nb_postes": data.get("nb_postes"),
        }

        # Boucle sur les postes
        for poste in data.get("postes", []):
            base_meta = {
                **base_meta_common,
                "poste_num": poste.get("poste_num"),
                "affectation": poste.get("affectation"),
                "groupe_fonction": poste.get("groupe_fonction"),
            }

            chunks = build_chunks_from_poste(base_meta, poste)
            documents.extend(chunks)

    except Exception as e:
        print(f"⚠️ Erreur JSON {filename} : {e}")

print(f"✅ Documents indexés : {len(documents)} chunks")


📂 Chargement des documents JSON structurés...
✅ Documents indexés : 592 chunks


In [ ]:
# =========================
# 5 — Embeddings & FAISS
# =========================

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
import torch

print("🔎 Initialisation du modèle d'embeddings...")

# 1️⃣ Chargement du modèle d'embeddings (léger et stable)
embedder = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device="cuda" if USE_GPU and torch.cuda.is_available() else "cpu"
)

# 2️⃣ Préparation des textes (sécurité)
texts = [d["text"] for d in documents if d.get("text")]

if not texts:
    raise ValueError("❌ Aucun texte valide pour la vectorisation.")

# 3️⃣ Encodage par batch (évite les pics mémoire)
print(f"📐 Calcul des embeddings pour {len(texts)} chunks...")

embeddings = embedder.encode(
    texts,
    batch_size=8 if USE_GPU else 4,      # 🔑 batch réduit pour stabilité
    normalize_embeddings=True,
    show_progress_bar=True
)

# 4️⃣ Conversion FAISS (float32 obligatoire)
embeddings = np.asarray(embeddings, dtype="float32")

# 5️⃣ Création de l'index FAISS (cosine similarity via IP)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

# Sécurité : vérifier cohérence
assert index.is_trained, "Index FAISS non entraîné"

# 6️⃣ Ajout des vecteurs
index.add(embeddings)

print(f"✅ Index FAISS créé avec {index.ntotal} vecteurs.")



🔎 Initialisation du modèle d'embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📐 Calcul des embeddings pour 592 chunks...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

✅ Index FAISS créé avec 592 vecteurs.


In [ ]:
def handle_small_talk(question: str):
    q = question.lower().strip()

    greetings = ["bonjour", "bonsoir", "salut", "hello", "hi"]
    thanks = ["merci", "merci beaucoup", "thanks"]

    if q in greetings:
        return (
            "Bonjour 👋\n"
            "Je suis l’agent d’information sur les concours ingénieurs du CNRS.\n"
            "Vous pouvez me poser des questions sur les concours, postes, missions ou compétences."
        )

    if q in thanks:
        return "Avec plaisir 😊 N’hésitez pas si vous avez d’autres questions sur les concours CNRS."

    return None


In [ ]:
from collections import Counter

def filter_same_source(chunks):
    """
    Garde uniquement les chunks provenant majoritairement
    du même fichier (même concours).
    """
    if not chunks:
        return []

    sources = [c["source"] for c in chunks]
    main_source = Counter(sources).most_common(1)[0][0]

    return [c for c in chunks if c["source"] == main_source]


In [ ]:
def is_orientation_question(question: str) -> bool:
    keywords = [
        "correspond", "profil", "je suis", "quel concours",
        "orienter", "adapté", "accessible avec"
    ]
    q = question.lower()
    return any(k in q for k in keywords)


In [ ]:
# =========================
# 6 — Retrieval avec reranking
# =========================

def retrieve(question: str):
    """
    Recherche les passages les plus pertinents pour une question donnée.
    Étapes :
    1. Retrieval large via FAISS (rappel)
    2. Filtrage par seuil de similarité
    3. Reranking précis (cross-encoder)
    4. Sélection finale TOP_K
    """

    #  Sécurité minimale
    if not question or len(question.strip()) < 3:
        return []

    # 1️⃣ Embedding de la question
    try:
        q_emb = embedder.encode(
            [question],
            normalize_embeddings=True
        )
    except Exception:
        return []

    # 2️⃣ Retrieval large (on sur-échantillonne pour le reranker)
    try:
        scores, indices = index.search(q_emb, TOP_K * 3)
    except Exception:
        return []

    # 3️⃣ Filtrage par similarité FAISS
    candidates = []
    for score, idx in zip(scores[0], indices[0]):

        # Index invalide
        if idx < 0 or idx >= len(documents):
            continue

        # Seuil anti-hallucination
        if score < SIMILARITY_THRESHOLD:
            continue

        doc = documents[idx].copy()
        doc["faiss_score"] = float(score)
        candidates.append(doc)

    # Aucun contexte fiable
    if not candidates:
        return []

    # 4️⃣ Reranking précis (si disponible)
    if reranker is not None:
        try:
            pairs = [(question, c["text"]) for c in candidates]
            rerank_scores = reranker.predict(pairs)

            for c, r_score in zip(candidates, rerank_scores):
                c["rerank_score"] = float(r_score)

            # Combinaison FAISS + reranker
            reranked = sorted(
                candidates,
                key=lambda c: (c["rerank_score"], c["faiss_score"]),
                reverse=True
            )
        except Exception:
            # Fallback : FAISS seul
            reranked = sorted(
                candidates,
                key=lambda c: c["faiss_score"],
                reverse=True
            )
    else:
        reranked = sorted(
            candidates,
            key=lambda c: c["faiss_score"],
            reverse=True
        )

    # 5️⃣ Sélection finale (TOP_K)
    return reranked[:TOP_K]


In [ ]:
# =========================
# 7 — Chargement du modèle LLM (Qwen/Qwen2.5-14B-Instruct)
# =========================

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import os

print("🚀 Chargement du modèle LLM...")

# 1️⃣ Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

# 2️⃣ Chargement du modèle avec gestion GPU / CPU
if USE_GPU and torch.cuda.is_available():

    print("🟢 GPU détecté — Chargement en mode GPU optimisé")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",                  # Placement automatique GPU / CPU
        torch_dtype=torch.float16,          # Réduction mémoire
        low_cpu_mem_usage=True,
        max_memory={
            0: "14GiB",                     # GPU T4 ≈ 16 Go
            "cpu": "10GiB"
        }
    )

else:
    print("🟡 GPU non disponible — Chargement en mode CPU (secours)")

    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    torch.set_num_threads(4)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )

# 3️⃣ Mise en mode évaluation (important)
model.eval()

# 4️⃣ Pipeline de génération STRICT (anti-hallucination)
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,

    # 🔑 CLÉ DU SUCCÈS
    do_sample=False,        # Toujours déterministe
    temperature=0.2,
    top_p=1.0,              # On ne tronque plus le raisonnement

    repetition_penalty=1.1, # Évite le blabla

    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

print("✅ Modèle prêt pour l'inférence.")


🚀 Chargement du modèle LLM...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Modèle prêt pour l'inférence.


In [ ]:
# =========================
# 8 — Prompt CNRS STRICT (anti-hallucination)
# =========================

def build_prompt(question: str, contexts: list) -> str:
    context_text = "\n".join(
        f"- {c['text']}"
        for c in contexts
    )

    return f"""
Tu es un agent officiel d'information sur les concours ingénieurs du CNRS.

RÈGLES STRICTES :
- Utilise UNIQUEMENT les informations présentes dans le CONTEXTE.
- Ne déduis rien.
- Ne complète rien.
- Ne donne aucune information absente du CONTEXTE.
- Si l'information demandée n'est pas clairement présente, répond EXACTEMENT :
  "{REFUS}"
- Répond uniquement en français.
- Ne pose pas de questions.

CONTEXTE :
{context_text}

QUESTION :
{question}

RÉPONSE :
"""
# #################################################################################

def build_orientation_prompt(question, contexts):
    context_text = "\n".join(
        f"- {c['text']}"
        for c in contexts
    )

    return f"""
Tu es un agent d'information du CNRS.

Tu dois aider à ORIENTER un candidat en comparant son profil
avec les missions et compétences décrites dans les documents.

RÈGLES :
- Tu ne dois PAS affirmer une correspondance certaine.
- Tu dois utiliser des formulations prudentes :
  "peut correspondre", "est cohérent avec", "semble en adéquation".
- Tu dois expliquer SUR QUELS ÉLÉMENTS tu te bases.
- Tu dois citer les sources utilisées.
- Si aucune correspondance raisonnable n'est possible, dis-le clairement.

DOCUMENTS :
{context_text}

QUESTION :
{question}

RÉPONSE :
"""

################################################################################
def build_search_query(question: str) -> str:
    """
    Transforme une question d'orientation en requête documentaire exploitable.
    """
    q = question.lower()

    keywords = []
    if "statistique" in q or "data" in q:
        keywords.append("analyse de données")
    if "biologique" in q or "bio" in q:
        keywords.append("biologie")
    if "informatique" in q:
        keywords.append("informatique")
    if "master" in q or "bac+5" in q:
        keywords.append("ingénieur")
    if "omics" in q:
        keywords.append("omiques")

    # fallback minimal
    if not keywords:
        return question

    return " ".join(keywords)


In [ ]:
# =========================
# 9 — Fonction answer()
# =========================
REFUS = "Je ne dispose pas de cette information dans les documents de référence."

def answer(question: str):
    # 1️⃣ Small talk
    st = handle_small_talk(question)
    if st:
        return {"response": st, "sources": [], "has_answer": True}

    # 2️⃣ Détection orientation
    orientation = is_orientation_question(question)

    # 3️⃣ Construction requête FAISS
    search_query = build_search_query(question) if orientation else question

    # 4️⃣ Retrieval
    contexts = retrieve(search_query)

    if not contexts:
        return {"response": REFUS, "sources": [], "has_answer": False}

    # 5️⃣ Filtrage par concours
    contexts = filter_same_source(contexts)

    # 6️⃣ Choix du prompt
    if orientation:
        prompt = build_orientation_prompt(question, contexts)
    else:
        prompt = build_prompt(question, contexts)

    # 7️⃣ Génération
    output = pipe(prompt)[0]["generated_text"].strip()

    # 8️⃣ Refus
    if REFUS.lower() in output.lower():
        return {"response": REFUS, "sources": [], "has_answer": False}

    # 9️⃣ Sources propres
    sources = sorted({f"{c['source']} | page {c['page']}" for c in contexts})

    return {"response": output, "sources": sources, "has_answer": True}


In [ ]:
def print_answer(result):
    print("\n📘 RÉPONSE :\n")
    print(result["response"])

    print("\n📂 SOURCES :")
    if not result["sources"]:
        print("Aucune")
    else:
        for s in result["sources"]:
            print(f"- {s}")


In [ ]:
def select_contexts_for_llm(contexts, max_chunks=3):
    """
    Sélectionne intelligemment les chunks à envoyer au LLM
    pour éviter les OOM et maximiser la pertinence.
    Priorité : Mission > Compétences > Contexte > Autres
    """
    if not contexts:
        return []

    priority_order = ["Mission", "Compétences", "Contexte", "Activités"]

    selected = []

    for label in priority_order:
        for c in contexts:
            if label.lower() in c["text"].lower() and c not in selected:
                selected.append(c)
                break
        if len(selected) >= max_chunks:
            break

    # Fallback : compléter si pas assez de chunks
    if len(selected) < max_chunks:
        for c in contexts:
            if c not in selected:
                selected.append(c)
            if len(selected) >= max_chunks:
                break

    return selected[:max_chunks]


In [ ]:
def answer(question: str):
    # 1️⃣ Small talk
    st = handle_small_talk(question)
    if st:
        return {
            "response": st,
            "sources": [],
            "has_answer": True
        }

    # 2️⃣ Détection orientation
    orientation = is_orientation_question(question)

    # 3️⃣ Requête de recherche adaptée
    search_query = build_search_query(question) if orientation else question

    # 4️⃣ Retrieval
    contexts = retrieve(search_query)

    if not contexts:
        return {
            "response": REFUS,
            "sources": [],
            "has_answer": False
        }

    # 5️⃣ 🔒 LIMITATION INTELLIGENTE DU CONTEXTE (ANTI-OOM)
    contexts = select_contexts_for_llm(
        contexts,
        max_chunks=MAX_CHUNKS_FOR_LLM
    )

    # 6️⃣ Construction du prompt
    if orientation:
        prompt = build_orientation_prompt(question, contexts)
    else:
        prompt = build_prompt(question, contexts)

    # 7️⃣ Génération
    try:
        output = pipe(prompt)[0]["generated_text"].strip()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return {
                "response": "Erreur mémoire GPU. Merci de reformuler la question.",
                "sources": [],
                "has_answer": False
            }
        raise e

    # 8️⃣ Détection refus
    if REFUS.lower() in output.lower():
        return {
            "response": REFUS,
            "sources": [],
            "has_answer": False
        }

    # 9️⃣ Sources propres et dédupliquées
    sources = sorted({
        f"{c['source']} | poste {c.get('poste_num')} | {c.get('affectation')}"
        for c in contexts
    })

    return {
        "response": output,
        "sources": list(sources),
        "has_answer": True
    }


In [ ]:
print("\n" + "=" * 70)
print("🤖 Agent conversationnel CNRS – Concours Ingénieurs")
print("📌 Basé uniquement sur les documents officiels fournis")
print("✋ Tapez 'stop', 'quit' ou 'exit' pour quitter")
print("=" * 70 + "\n")

while True:
    try:
        q = input("❓ Votre question : ").strip()

        if not q:
            print("⚠️ Merci de poser une question.\n")
            continue

        if q.lower() in ["stop", "quit", "exit"]:
            print("\n👋 Fin de la session. Merci !")
            break

        print("\n🔎 Analyse de la question...\n")

        # 🔹 APPEL CORRECT DE answer()
        result = answer(q)

        # 🔹 AFFICHAGE PROPRE
        print_answer(result)

        print("\n" + "-" * 70)

    except KeyboardInterrupt:
        print("\n\n👋 Interruption utilisateur. Fin de la session.")
        break

    except Exception as e:
        print(f"\n❌ Erreur inattendue : {e}")
        print("\n" + "-" * 70)


🤖 Agent conversationnel CNRS – Concours Ingénieurs
📌 Basé uniquement sur les documents officiels fournis
✋ Tapez 'stop', 'quit' ou 'exit' pour quitter



❓ Votre question :  Helle



🔎 Analyse de la question...


📘 RÉPONSE :

Je ne dispose pas de cette information dans les documents de référence.

📂 SOURCES :
Aucune

----------------------------------------------------------------------


❓ Votre question :  stop



👋 Fin de la session. Merci !


# Metrique

In [ ]:
import torch
from transformers import pipeline

judge_pipe = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    #model = "google/flan-t5-large",
    device=0 if torch.cuda.is_available() else -1
)



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [3]:
import pandas as pd

data = pd.read_csv("/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/jeu_test.csv")

data.head()


,ID,Question,expected_answer,category,must_refuse,must_cite_source,expected_sources
0,1,Combien de postes sont disponibles pour le con...,Le concours comprend trois postes distincts :\...,concours_info,0.0,1.0,page_001.html
1,2,Est ce que le concours Experte ou expert en in...,"Pour le concours n°64, le grade mentionné est ...",orientation,0.0,1.0,Guide candidat 2025.pdf
2,3,Quelles sont les responsabilités de l'ingénieu...,L'agent développera des outils numériques avan...,concours_info,0.0,1.0,page_064.html
3,4,Quel salaire je peux prétendre en fin de carri...,Le concours numéro 64 fait référence à un pos...,career_general,0.0,1.0,Guide candidat 2025.pdf
4,5,Est ce que le lieu du concours Experte ou expe...,La fiche du concours indique que le lieu de tr...,process_general,1.0,0.0,NaN


# Fidélité / Faithfulness (LLM-as-Judge)

In [ ]:
import numpy as np
import pandas as pd

def evaluate_faithfulness(answer, contexts, judge_pipe, max_context_chars=2000):
    context_text = " ".join((c.get("text", "") or "") for c in contexts)[:max_context_chars]

    prompt = f"""
Cette réponse est-elle fidèle au contexte ?
Réponds uniquement par OUI ou NON.

Contexte :
{context_text}

Réponse :
{answer}

La réponse est-elle soutenue par le contexte ?
"""

    out = judge_pipe(prompt, max_new_tokens=5, do_sample=False)[0]["generated_text"].lower().strip()
    return 1 if out.startswith("oui") else 0


faithfulness_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    ans_dict = answer(q)
    ans_only = ans_dict["response"]

    contexts = retrieve(q)

    score = evaluate_faithfulness(
        answer=ans_only,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    faithfulness_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Faithfulness": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("faithfulness_results_qwen.csv", index=False)

print("✅ Fidélité moyenne :", np.mean(faithfulness_scores))
results_df.head()


Token indices sequence length is longer than the specified maximum sequence length for this model (684 > 512). Running this sequence through the model will result in indexing errors
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Fidélité moyenne : 0.0


,ID,Question,Faithfulness
0,1,Combien de postes sont disponibles pour le con...,0
1,2,Est ce que le concours Experte ou expert en in...,0
2,3,Quelles sont les responsabilités de l'ingénieu...,0
3,4,Quel salaire je peux prétendre en fin de carri...,0
4,5,Est ce que le lieu du concours Experte ou expe...,0


# 2) Pertinence Answer Relevancy (LLM-as-Judge 1→5)

In [ ]:
import re
import numpy as np
import pandas as pd

def evaluate_answer_relevancy(question, answer, judge_pipe, max_new_tokens=10):

    prompt = f"""
Donne une note de 1 à 5 uniquement (un seul chiffre).

Question :
{question}

Réponse :
{answer}

Pertinence (1 = hors sujet, 5 = répond parfaitement) :
"""

    out = judge_pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )[0]["generated_text"].strip()

    match = re.search(r"\b([1-5])\b", out)
    if not match:
        return 0.0

    score = int(match.group(1))
    return score / 5.0


relevancy_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    ans_dict = answer(q)
    ans_only = ans_dict["response"]

    score = evaluate_answer_relevancy(
        question=q,
        answer=ans_only,
        judge_pipe=judge_pipe
    )

    relevancy_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Answer_Relevancy_Judge": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("answer_relevancy_judge_results_qwen.csv", index=False)

print("✅ Pertinence moyenne :", np.mean(relevancy_scores))
results_df.head()


✅ Pertinence moyenne : 0.19999999999999998


,ID,Question,Answer_Relevancy_Judge
0,1,Combien de postes sont disponibles pour le con...,0.2
1,2,Est ce que le concours Experte ou expert en in...,0.2
2,3,Quelles sont les responsabilités de l'ingénieu...,0.2
3,4,Quel salaire je peux prétendre en fin de carri...,0.2
4,5,Est ce que le lieu du concours Experte ou expe...,0.2


# 3) Context Precision (LLM-as-Judge)

In [ ]:
import numpy as np
import pandas as pd

def evaluate_context_precision(question, contexts, judge_pipe, max_chunk_chars=800):
    if not contexts:
        return 0.0

    relevant = 0

    for c in contexts:
        chunk_text = (c.get("text", "") or "")[:max_chunk_chars]

        prompt = f"""
Réponds uniquement par OUI ou NON.

Question :
{question}

Contexte :
{chunk_text}

Ce contexte est-il pertinent pour répondre à la question ?
"""

        out = judge_pipe(
            prompt,
            max_new_tokens=3,
            do_sample=False
        )[0]["generated_text"].lower().strip()

        if out.startswith("oui"):
            relevant += 1

    return relevant / len(contexts)


context_precision_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]
    contexts = retrieve(q)

    score = evaluate_context_precision(
        question=q,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    context_precision_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Context_Precision": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("context_precision_results_qwen.csv", index=False)

print("✅ Context Precision moyenne :", np.mean(context_precision_scores))
results_df.head()


✅ Context Precision moyenne : 0.0026041666666666665


,ID,Question,Context_Precision
0,1,Combien de postes sont disponibles pour le con...,0.0
1,2,Est ce que le concours Experte ou expert en in...,0.0
2,3,Quelles sont les responsabilités de l'ingénieu...,0.0
3,4,Quel salaire je peux prétendre en fin de carri...,0.0
4,5,Est ce que le lieu du concours Experte ou expe...,0.0


# 4) Context Recall (LLM-as-Judge + expected_answer du CSV)

In [ ]:
import numpy as np
import pandas as pd

def evaluate_context_recall(ground_truth_answer, contexts, judge_pipe, max_context_chars=1500):
    if not contexts:
        return 0

    context_text = " ".join(
        (c.get("text", "") or "")[:400] for c in contexts
    )[:max_context_chars]

    prompt = f"""
Réponds uniquement par OUI ou NON.

Règle stricte :
- OUI = toutes les informations essentielles sont présentes dans le contexte
- NON = s'il manque au moins une information essentielle

Contexte :
{context_text}

Réponse attendue :
{ground_truth_answer}

Toutes les informations nécessaires pour produire EXACTEMENT cette réponse
sont-elles présentes dans le contexte ?
"""

    out = judge_pipe(
        prompt,
        max_new_tokens=3,
        do_sample=False
    )[0]["generated_text"].lower().strip()

    return 1 if out.startswith("oui") else 0


context_recall_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]
    gt = row["expected_answer"]

    contexts = retrieve(q)

    score = evaluate_context_recall(
        ground_truth_answer=gt,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    context_recall_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Context_Recall": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("context_recall_results_qwen.csv", index=False)

print("✅ Context Recall moyen :", np.mean(context_recall_scores))
results_df.head()


✅ Context Recall moyen : 0.0


,ID,Question,Context_Recall
0,1,Combien de postes sont disponibles pour le con...,0
1,2,Est ce que le concours Experte ou expert en in...,0
2,3,Quelles sont les responsabilités de l'ingénieu...,0
3,4,Quel salaire je peux prétendre en fin de carri...,0
4,5,Est ce que le lieu du concours Experte ou expe...,0


# 5) Refusal Accuracy (avec must_refuse du CSV)

In [ ]:
import numpy as np
import pandas as pd
import re

REFUSAL_PATTERNS = [
    r"je ne dispose pas",
    r"je n'ai pas (?:cette|ces) information",
    r"information .*pas .*dans les documents",
    r"aucune (?:source|information)",
    r"je ne peux pas répondre",
]

def is_refusal(text: str) -> bool:
    if not text:
        return True
    t = text.lower().strip()
    return any(re.search(p, t) for p in REFUSAL_PATTERNS)


all_results = []
correct_list = []

for idx, row in data.iterrows():
    q = row["Question"]
    must_refuse = int(row["must_refuse"]) if pd.notna(row["must_refuse"]) else 0

    ans_dict = answer(q)
    ans_only = ans_dict["response"]

    gt_refusal = (must_refuse == 1)
    ans_refusal = is_refusal(ans_only)

    correct = int(gt_refusal == ans_refusal)
    correct_list.append(correct)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "must_refuse": must_refuse,
        "ans_refusal": int(ans_refusal),
        "Refusal_Correct": correct
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("refusal_accuracy_results_qwen.csv", index=False)

print("✅ Refusal Accuracy :", np.mean(correct_list))
results_df.head()


✅ Refusal Accuracy : 0.7291666666666666


,ID,Question,must_refuse,ans_refusal,Refusal_Correct
0,1,Combien de postes sont disponibles pour le con...,0,0,1
1,2,Est ce que le concours Experte ou expert en in...,0,0,1
2,3,Quelles sont les responsabilités de l'ingénieu...,0,0,1
3,4,Quel salaire je peux prétendre en fin de carri...,0,0,1
4,5,Est ce que le lieu du concours Experte ou expe...,1,0,0


# 6) Retrieval Precision@k / Recall@k / F1@k (avec expected_sources)

In [ ]:
import numpy as np
import pandas as pd
import re
import os

def precision_at_k(retrieved, relevant, k):
    top_k = set(retrieved[:k])
    relevant_retrieved = top_k & relevant
    return len(relevant_retrieved) / k if k > 0 else 0

def recall_at_k(retrieved, relevant, k):
    top_k = set(retrieved[:k])
    relevant_retrieved = top_k & relevant
    return len(relevant_retrieved) / len(relevant) if relevant else 0

def f1_at_k(p, r):
    if (p + r) == 0:
        return 0.0
    return 2 * p * r / (p + r)

def parse_expected_sources(x):
    if pd.isna(x):
        return set()
    parts = re.split(r"[;,]", str(x))
    return set(p.strip() for p in parts if p.strip())

def normalize_source_name(filename: str):
    if not filename:
        return ""
    base = os.path.basename(str(filename)).strip()
    return os.path.splitext(base)[0]


K = TOP_K

precision_scores = []
recall_scores = []
f1_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    expected = parse_expected_sources(row["expected_sources"])
    expected_norm = set(normalize_source_name(s) for s in expected)

    contexts = retrieve(q)
    retrieved_sources = [c.get("source", "") for c in contexts]
    retrieved_norm = [normalize_source_name(s) for s in retrieved_sources]

    # doc-level unique
    retrieved_unique = []
    seen = set()
    for s in retrieved_norm:
        if s not in seen:
            retrieved_unique.append(s)
            seen.add(s)

    p = precision_at_k(retrieved_unique, expected_norm, k=K)
    r = recall_at_k(retrieved_unique, expected_norm, k=K)
    f1 = f1_at_k(p, r)

    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        f"Precision@{K}": p,
        f"Recall@{K}": r,
        f"F1@{K}": f1,
        "expected_sources": str(row["expected_sources"]),
        "retrieved_sources": ", ".join(retrieved_sources)
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv(f"retrieval_metrics_at_{K}_qwen.csv", index=False)

print(f"✅ Precision@{K} moyenne :", np.mean(precision_scores))
print(f"✅ Recall@{K} moyen :", np.mean(recall_scores))
print(f"✅ F1@{K} moyen :", np.mean(f1_scores))

results_df.head()


✅ Precision@8 moyenne : 0.044270833333333336
✅ Recall@8 moyen : 0.3541666666666667
✅ F1@8 moyen : 0.0787037037037037


,ID,Question,Precision@8,Recall@8,F1@8,expected_sources,retrieved_sources
0,1,Combien de postes sont disponibles pour le con...,0.125,1.0,0.222222,page_001.html,"page_001.json, page_001.json, page_004.json, p..."
1,2,Est ce que le concours Experte ou expert en in...,0.000,0.0,0.000000,Guide candidat 2025.pdf,"page_071.json, page_065.json, page_073.json, p..."
2,3,Quelles sont les responsabilités de l'ingénieu...,0.000,0.0,0.000000,page_064.html,"page_059.json, page_087.json, page_055.json, p..."
3,4,Quel salaire je peux prétendre en fin de carri...,0.000,0.0,0.000000,Guide candidat 2025.pdf,"page_080.json, page_077.json, page_080.json, p..."
4,5,Est ce que le lieu du concours Experte ou expe...,0.000,0.0,0.000000,nan,"page_096.json, page_046.json, page_046.json, p..."


# 7) Answer Similarity (generated vs expected_answer) (Embeddings)

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import util

def answer_similarity(generated_answer, expected_answer, embedder):
    gen_emb = embedder.encode(generated_answer, normalize_embeddings=True)
    exp_emb = embedder.encode(expected_answer, normalize_embeddings=True)
    return float(util.cos_sim(gen_emb, exp_emb).item())


sim_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]
    gt = row["expected_answer"]

    ans_dict = answer(q)
    ans_only = ans_dict["response"]

    score = answer_similarity(
        generated_answer=ans_only,
        expected_answer=gt,
        embedder=embedder
    )

    sim_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Answer_Similarity_GT": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("answer_similarity_gt_qwen.csv", index=False)

print("✅ Similarité moyenne ANS vs GT :", np.mean(sim_scores))
results_df.head()


✅ Similarité moyenne ANS vs GT : 0.27812888969977695


,ID,Question,Answer_Similarity_GT
0,1,Combien de postes sont disponibles pour le con...,0.114580
1,2,Est ce que le concours Experte ou expert en in...,0.222967
2,3,Quelles sont les responsabilités de l'ingénieu...,0.078520
3,4,Quel salaire je peux prétendre en fin de carri...,0.479011
4,5,Est ce que le lieu du concours Experte ou expe...,0.244559


# 8) Answer Relevancy (Embeddings) (Question vs Answer)

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import util

def answer_relevancy_embedding(question, answer, embedder):
    q_emb = embedder.encode(question, normalize_embeddings=True)
    a_emb = embedder.encode(answer, normalize_embeddings=True)
    return float(util.cos_sim(q_emb, a_emb).item())


scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    ans_dict = answer(q)
    ans_only = ans_dict["response"]

    score = answer_relevancy_embedding(
        question=q,
        answer=ans_only,
        embedder=embedder
    )

    scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Answer_Relevancy_Embedding": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("answer_relevancy_embedding_qwen.csv", index=False)

print("✅ Relevancy (embedding) moyenne :", np.mean(scores))
results_df.head()


✅ Relevancy (embedding) moyenne : 0.2933392468839884


,ID,Question,Answer_Relevancy_Embedding
0,1,Combien de postes sont disponibles pour le con...,0.284108
1,2,Est ce que le concours Experte ou expert en in...,0.208525
2,3,Quelles sont les responsabilités de l'ingénieu...,0.264365
3,4,Quel salaire je peux prétendre en fin de carri...,0.395033
4,5,Est ce que le lieu du concours Experte ou expe...,0.299135
